## To prepare a consume the resulting images from the blue print action on Unreal
* Two mask and one image generated, 
    * OG, the original image, that will be used to train the YOLO model
    * BLUE, and X-ray like image identifying where the object is
    * RED, a second X-ray, but with occlusion from the objects on from of the target shoe
    
* Why two x-rays, the blue tells me the total are in the picture that the shoe occupies, if no other object was present, the red, shoes the area that is actually visible, from this i can calculate a visibility score and ignore all images where this score is to low. 
* The red x-ray will allow me to calculate the box for supervised training


<table>
  <tr>
    <td><img src="Images/image_0CE3ADA94EC4E001DF1744A11387B4D0_BLUE.png" alt="BLUE" width="100%"></td>
    <td><img src="Images/image_0CE3ADA94EC4E001DF1744A11387B4D0_OG.png" alt="OG" width="100%"></td>
    <td><img src="Images/image_0CE3ADA94EC4E001DF1744A11387B4D0_RED.png" alt="RED" width="100%"></td>
  </tr>
</table>

In [1]:
"""
prepare_dataset.py

Processes a folder of masked images into a YOLO-ready dataset.

Expected input structure:
    input_dir/
        <name>_OG.png    original image
        <name>_RED.png   mask: visible portion of object (red pixels)
        <name>_BLUE.png  mask: full object extent (blue pixels)

Output structure:
    output_dir/
        images/   original images
        labels/   YOLO .txt labels (empty or with bounding box)

Visibility = red_area / blue_area
    < 20%   → copy image + empty label  (object not present)
    20-60%  → skip                      (ambiguous, not used)
    > 60%   → copy image + YOLO label   (bounding box from RED mask)
"""

'\nprepare_dataset.py\n\nProcesses a folder of masked images into a YOLO-ready dataset.\n\nExpected input structure:\n    input_dir/\n        <name>_OG.png    original image\n        <name>_RED.png   mask: visible portion of object (red pixels)\n        <name>_BLUE.png  mask: full object extent (blue pixels)\n\nOutput structure:\n    output_dir/\n        images/   original images\n        labels/   YOLO .txt labels (empty or with bounding box)\n\nVisibility = red_area / blue_area\n    < 20%   → copy image + empty label  (object not present)\n    20-60%  → skip                      (ambiguous, not used)\n    > 60%   → copy image + YOLO label   (bounding box from RED mask)\n'

In [2]:
import os
import shutil
import numpy as np
from pathlib import Path
from PIL import Image


# ── Configuration ─────────────────────────────────────────────────────────────

INPUT_DIR  = Path("input")       # folder with OG / RED / BLUE images
OUTPUT_DIR = Path("output")      # destination

VISIBILITY_LOW  = 0.20           # below → empty label
VISIBILITY_HIGH = 0.60           # above → yolo label  (between → skip) 

YOLO_CLASS = 0

In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def red_pixels(img_array: np.ndarray) -> np.ndarray:
    """Boolean mask of pixels that are 'red' in an RGB image."""
    r, g, b = img_array[..., 0], img_array[..., 1], img_array[..., 2]
    return (r > 127) & (g < 80) & (b < 80)


def blue_pixels(img_array: np.ndarray) -> np.ndarray:
    """Boolean mask of pixels that are 'blue' in an RGB image."""
    r, g, b = img_array[..., 0], img_array[..., 1], img_array[..., 2]
    return (b > 127) & (r < 80) & (g < 80)


def bounding_box_yolo(mask: np.ndarray, img_h: int, img_w: int) -> tuple:
    """
    Returns YOLO-format bounding box (x_center, y_center, width, height)
    normalised to [0, 1] from a boolean pixel mask.
    """
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    y_min, y_max = np.where(rows)[0][[0, -1]]
    x_min, x_max = np.where(cols)[0][[0, -1]]

    x_center = ((x_min + x_max) / 2) / img_w
    y_center = ((y_min + y_max) / 2) / img_h
    width    = (x_max - x_min) / img_w
    height   = (y_max - y_min) / img_h

    return x_center, y_center, width, height


def ensure_dirs(output: Path):
    (output / "images").mkdir(parents=True, exist_ok=True)
    (output / "labels").mkdir(parents=True, exist_ok=True)

In [4]:
# ── Main ──────────────────────────────────────────────────────────────────────

def process(input_dir: Path, output_dir: Path):
    ensure_dirs(output_dir)

    og_files = sorted(input_dir.glob("*_OG.png"))

    if not og_files:
        print(f"No *_OG.png files found in {input_dir}")
        return

    stats = {"copied_empty": 0, "copied_label": 0, "skipped": 0, "errors": 0}

    for og_path in og_files:
        stem = og_path.stem[: -len("_OG")]          # strip trailing _OG
        red_path  = input_dir / f"{stem}_RED.png"
        blue_path = input_dir / f"{stem}_BLUE.png"

        # ── Validate all three files exist ────────────────────────────────────
        missing = [p for p in (red_path, blue_path) if not p.exists()]
        if missing:
            print(f"[WARN] {stem}: missing {[p.name for p in missing]}, skipping.")
            stats["errors"] += 1
            continue

        # ── Load masks ────────────────────────────────────────────────────────
        try:
            og_img   = Image.open(og_path).convert("RGB")
            red_img  = Image.open(red_path).convert("RGB")
            blue_img = Image.open(blue_path).convert("RGB")
        except Exception as e:
            print(f"[ERROR] {stem}: {e}")
            stats["errors"] += 1
            continue

        img_h, img_w = og_img.size[1], og_img.size[0]

        red_arr  = np.array(red_img)
        blue_arr = np.array(blue_img)

        red_mask  = red_pixels(red_arr)
        blue_mask = blue_pixels(blue_arr)

        red_area  = int(red_mask.sum())
        blue_area = int(blue_mask.sum())

        # ── Visibility ────────────────────────────────────────────────────────
        if blue_area == 0:
            # No object in blue mask at all → treat as not present
            visibility = 0.0
        else:
            visibility = red_area / blue_area

      

        # ── Routing logic ─────────────────────────────────────────────────────
        if visibility < VISIBILITY_LOW:
            # Object not meaningfully visible → copy with empty label
            dest_image = output_dir / "images" / f"empty_{og_path.name}"
            dest_label = output_dir / "labels" / f"empty_{stem}_OG.txt"
        
            shutil.copy2(og_path, dest_image)
            dest_label.write_text("")
            stats["copied_empty"] += 1
            print(f"[EMPTY ] {stem}  vis={visibility:.2%}  red={red_area}px  blue={blue_area}px")

        elif visibility > VISIBILITY_HIGH:
            # Object clearly visible → copy with YOLO bounding box from RED mask
            if red_mask.sum() == 0:
                print(f"[WARN] {stem}: visibility={visibility:.2%} but red mask is empty, skipping.")
                stats["errors"] += 1
                continue

            dest_image = output_dir / "images" / og_path.name
            dest_label = output_dir / "labels" / f"{stem}_OG.txt"

            x_c, y_c, w, h = bounding_box_yolo(red_mask, img_h, img_w)
            shutil.copy2(og_path, dest_image)
            dest_label.write_text(f"{YOLO_CLASS} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")
            stats["copied_label"] += 1
            print(f"[LABEL ] {stem}  vis={visibility:.2%}  bbox=({x_c:.3f},{y_c:.3f},{w:.3f},{h:.3f})")

        else:
            # Ambiguous visibility → skip
            stats["skipped"] += 1
            print(f"[SKIP  ] {stem}  vis={visibility:.2%}  (between {VISIBILITY_LOW:.0%}–{VISIBILITY_HIGH:.0%})")

    # ── Summary ───────────────────────────────────────────────────────────────
    total     = sum(stats.values())
    processed = total - stats["errors"]

    def bar(count, width=40) -> str:
        filled = round((count / total) * width) if total else 0
        return "█" * filled + "░" * (width - filled)

    def pct(count) -> str:
        return f"{count / total * 100:5.1f}%" if total else "  0.0%"

    print(f"""
╔══════════════════════════════════════════════════════╗
║           Dataset Distribution — {total:>4} triplets        ║
╠══════════════════════════════════════════════════════╣
║  Class 0  (vis > {VISIBILITY_HIGH:.0%})  {bar(stats['copied_label'])}  {pct(stats['copied_label'])}  ({stats['copied_label']})
║  Empty    (vis < {VISIBILITY_LOW:.0%})  {bar(stats['copied_empty'])}  {pct(stats['copied_empty'])}  ({stats['copied_empty']})
║  Ignored  ({VISIBILITY_LOW:.0%}–{VISIBILITY_HIGH:.0%})    {bar(stats['skipped'])}  {pct(stats['skipped'])}  ({stats['skipped']})
║  Errors                   {bar(stats['errors'])}  {pct(stats['errors'])}  ({stats['errors']})
╠══════════════════════════════════════════════════════╣
║  Copied to output: {stats['copied_label'] + stats['copied_empty']:>4}  │  Discarded: {stats['skipped']:>4}  │  Errors: {stats['errors']:>3}  ║
╚══════════════════════════════════════════════════════╝""")


In [5]:
INPUT_DIR = Path("C:/Temp/img")
OUTPUT_DIR = Path("./Datasets/Original/Outside_2")

process(INPUT_DIR, OUTPUT_DIR)

[LABEL ] image_01096E6B426CAEBE1508179269C2DE7B  vis=99.95%  bbox=(0.499,0.498,0.027,0.133)
[LABEL ] image_0123C1E94DCDDDDA050788AA3FC41A62  vis=99.58%  bbox=(0.500,0.503,0.074,0.068)
[LABEL ] image_03964C594DD3A4293EE790B44431A644  vis=99.78%  bbox=(0.504,0.502,0.045,0.129)
[LABEL ] image_03A4012242B009EB79682A8D637D5828  vis=70.49%  bbox=(0.491,0.498,0.066,0.071)
[LABEL ] image_043FD04B409363518F0F6D8FAE69F0A5  vis=99.84%  bbox=(0.499,0.494,0.024,0.087)
[LABEL ] image_0458761440C6C0F444B6378A489D7E20  vis=100.03%  bbox=(0.503,0.500,0.035,0.109)
[LABEL ] image_04EE022C4B956C63E0104B823A183119  vis=87.43%  bbox=(0.497,0.494,0.036,0.083)
[EMPTY ] image_04EEC7DD432C98FFC446EB8AF766AD61  vis=8.89%  red=370px  blue=4161px
[SKIP  ] image_05160C6B467437DD9622049238CF299F  vis=51.34%  (between 20%–60%)
[LABEL ] image_058721A642F4558023EB2E9B33C13C55  vis=99.76%  bbox=(0.499,0.489,0.079,0.069)
[LABEL ] image_05B29ABA4E794A3D761E1AB09A7B34D9  vis=99.91%  bbox=(0.502,0.502,0.063,0.071)
[LABEL ] 

V2

In [6]:
"""
mask_to_yolo.py  (v5 - lida com oclusao que parte o objeto em varios pedacos)
Le os screenshots do Unreal (pares RGB + mascara) e gera as labels YOLO.

NOVIDADE v5: quando o objeto esta partido em varios pedacos pela oclusao
(ex: sapatilha cortada ao meio pelo poste), a bounding box engloba TODOS
os pedacos, nao apenas o maior. Pequenos pontos de ruido isolado sao
descartados (componentes abaixo de MIN_COMPONENT pixels).

Processa UM cenario de cada vez (com prefixo no nome).

COMO USAR:
  1. Ajusta SCENARIO_DIR e PREFIXO em baixo
  2. python mask_to_yolo.py
  3. Muda para o outro cenario e corre outra vez

Requisitos: pip install pillow numpy scipy
"""

import os
import glob
import shutil
import numpy as np
from PIL import Image

try:
    from scipy import ndimage
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

# ===================== AJUSTA AQUI =====================
SCENARIO_DIR = "c:/Temp/img"
PREFIXO      = "natureza"      # muda para "natureza" na 2a corrida

OUTPUT_DIR = "./Datasets/Original/Outside_1"
CLASS_ID = 0
# =======================================================

# --- Parametros de cor da mascara (laranja do Unreal) ---
R_MIN, G_MAX, B_MAX = 200, 140, 90
RG_DIFF, RB_DIFF = 80, 140

# Area TOTAL minima de laranja para considerar que ha sapatilha.
# Abaixo disto -> hard negative (label vazia).
MIN_PIXELS = 300

# Tamanho minimo de um componente para CONTAR para a bounding box.
# Componentes mais pequenos que isto sao considerados ruido e ignorados.
# (mas varios componentes validos sao TODOS englobados na mesma caixa)
MIN_COMPONENT = 80
# --------------------------------------------------------

OUT_IMAGES = os.path.join(OUTPUT_DIR, "images")
OUT_LABELS = os.path.join(OUTPUT_DIR, "labels")


def orange_mask(arr):
    r = arr[:, :, 0].astype(int)
    g = arr[:, :, 1].astype(int)
    b = arr[:, :, 2].astype(int)
    return ((r >= R_MIN) & (g <= G_MAX) & (b <= B_MAX) &
            ((r - g) >= RG_DIFF) & ((r - b) >= RB_DIFF))


def bbox_from_arr(arr):
    m = orange_mask(arr)
    total = m.sum()
    if total < MIN_PIXELS:
        return None

    if HAS_SCIPY:
        # Remove componentes pequenos (ruido), mas MANTEM todos os
        # componentes validos (a sapatilha pode estar partida pelo poste).
        labeled, n = ndimage.label(m)
        if n >= 1:
            keep = np.zeros_like(m)
            for comp_id in range(1, n + 1):
                comp = (labeled == comp_id)
                if comp.sum() >= MIN_COMPONENT:
                    keep |= comp
            m = keep
        if m.sum() < MIN_PIXELS:
            return None

    ys, xs = np.where(m)
    if len(xs) == 0:
        return None
    # A caixa engloba TODOS os pedacos validos (do extremo esquerdo
    # ao direito, do topo a base) -> cobre a sapatilha mesmo partida.
    return (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))


def to_yolo(bbox, w, h):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2 / w, (y1 + y2) / 2 / h,
            (x2 - x1) / w, (y2 - y1) / h)


def main():
    if not HAS_SCIPY:
        print("[aviso] scipy nao instalado - deteccao de ruido limitada.")
        print("        pip install scipy\n")

    os.makedirs(OUT_IMAGES, exist_ok=True)
    os.makedirs(OUT_LABELS, exist_ok=True)

    if not os.path.isdir(SCENARIO_DIR):
        print(f"[!] Pasta nao encontrada: {SCENARIO_DIR}")
        return

    files = sorted(glob.glob(os.path.join(SCENARIO_DIR, "*.png")))
    if not files:
        print(f"[!] Nenhum .png em {SCENARIO_DIR}")
        return

    print(f"[i] Cenario: '{PREFIXO}'  |  {len(files)} ficheiros")
    if len(files) % 2 != 0:
        print("[AVISO] Numero impar de ficheiros - o ultimo fica sem par!\n")

    existentes = glob.glob(os.path.join(OUT_IMAGES, f"{PREFIXO}_*.png"))
    start = len(existentes)

    pair = saved = negs = 0
    for i in range(0, len(files) - 1, 2):
        a = np.array(Image.open(files[i]).convert("RGB"))
        b = np.array(Image.open(files[i + 1]).convert("RGB"))

        if orange_mask(a).sum() >= orange_mask(b).sum():
            mask_arr, rgb_path = a, files[i + 1]
        else:
            mask_arr, rgb_path = b, files[i]

        h, w = mask_arr.shape[:2]
        bbox = bbox_from_arr(mask_arr)

        pair += 1
        base = f"{PREFIXO}_{start + pair:04d}"
        shutil.copy(rgb_path, os.path.join(OUT_IMAGES, base + ".png"))
        lbl = os.path.join(OUT_LABELS, base + ".txt")

        if bbox is None:
            open(lbl, "w").close()
            negs += 1
            continue

        xc, yc, bw, bh = to_yolo(bbox, w, h)
        with open(lbl, "w") as f:
            f.write(f"{CLASS_ID} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
        saved += 1

    print(f"\n  Pares processados : {pair}")
    print(f"  Com sapatilha     : {saved}")
    print(f"  Hard negatives    : {negs}")
    print(f"  Adicionados como  : {PREFIXO}_{start+1:04d} ... {PREFIXO}_{start+pair:04d}")
    print(f"  Imagens -> {OUT_IMAGES}")
    print(f"  Labels  -> {OUT_LABELS}")
    print("\n  Valida com: python validar.py")


if __name__ == "__main__":
    main()

[i] Cenario: 'natureza'  |  536 ficheiros

  Pares processados : 268
  Com sapatilha     : 253
  Hard negatives    : 15
  Adicionados como  : natureza_0001 ... natureza_0268
  Imagens -> ./Datasets/Original/Outside_1\images
  Labels  -> ./Datasets/Original/Outside_1\labels

  Valida com: python validar.py


In [7]:
"""
validar.py
Desenha as bounding boxes sobre as imagens RGB do dataset, para validacao visual.
Gera imagens novas numa pasta 'validacao/' com a caixa verde desenhada.

Uso:
  python validar.py
"""
import os
import glob
import random
from PIL import Image, ImageDraw

# Pastas (devem coincidir com o que o mask_to_yolo.py criou)
IMAGES_DIR = "./Datasets/Original/Outside_1/images"
LABELS_DIR = "./Datasets/Original/Outside_1/labels"
OUT_DIR = "./validacao"

# Quantas imagens validar (ao acaso). Mete um numero alto para ver todas.
QUANTAS = 1000

os.makedirs(OUT_DIR, exist_ok=True)

label_files = sorted(glob.glob(os.path.join(LABELS_DIR, "*.txt")))
if not label_files:
    print("[!] Nenhuma label encontrada em", LABELS_DIR)
    raise SystemExit

# separa as que tem caixa das vazias (hard negatives)
com_caixa = [p for p in label_files if os.path.getsize(p) > 0]
vazias = [p for p in label_files if os.path.getsize(p) == 0]

print(f"Total de labels   : {len(label_files)}")
print(f"Com sapatilha     : {len(com_caixa)}")
print(f"Hard negatives    : {len(vazias)}")

amostra = random.sample(com_caixa, min(QUANTAS, len(com_caixa)))

for lbl in amostra:
    base = os.path.splitext(os.path.basename(lbl))[0]
    img_path = os.path.join(IMAGES_DIR, base + ".png")
    if not os.path.exists(img_path):
        continue
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    with open(lbl) as f:
        parts = f.read().split()
    _, xc, yc, bw, bh = parts
    xc, yc, bw, bh = float(xc), float(yc), float(bw), float(bh)
    x1 = (xc - bw/2) * W; y1 = (yc - bh/2) * H
    x2 = (xc + bw/2) * W; y2 = (yc + bh/2) * H
    d = ImageDraw.Draw(img)
    d.rectangle([x1, y1, x2, y2], outline=(0, 255, 0), width=6)
    img.save(os.path.join(OUT_DIR, base + "_check.png"))

print(f"\n{len(amostra)} imagens de validacao criadas em: {OUT_DIR}")
print("Abre essa pasta e confirma que as caixas verdes envolvem bem a sapatilha.")

Total de labels   : 268
Com sapatilha     : 253
Hard negatives    : 15


KeyboardInterrupt: 